In [1]:
"""
Feature Engineering step in the diagram (50+ features -> select 20-30).
Pure pandas/numpy, no ta-lib dependency required.
"""
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# ค่าคงที่ในไฟล์นี้เอง ไม่ต้อง import config แล้ว
N_FEATURES_SELECTED = 25
TARGET_HORIZON = 1
RANDOM_STATE = 42


def _rsi(close: pd.Series, period: int) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def _atr(df: pd.DataFrame, period: int) -> pd.Series:
    hl = df["High"] - df["Low"]
    hc = (df["High"] - df["Close"].shift()).abs()
    lc = (df["Low"] - df["Close"].shift()).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    return tr.rolling(period).mean()


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """df must have Open/High/Low/Close/Volume indexed by time.
    Returns a new DataFrame of engineered features (raw OHLCV dropped)."""
    f = pd.DataFrame(index=df.index)
    close, high, low, vol = df["Close"], df["High"], df["Low"], df["Volume"]

    # --- returns & momentum at multiple lags (12 features) ---
    for lag in [1, 2, 3, 5, 8, 13, 21, 34]:
        f[f"ret_{lag}"] = close.pct_change(lag)
    for lag in [1, 3, 5, 10]:
        f[f"mom_{lag}"] = close - close.shift(lag)

    # --- explicit lag features: 1/3/5/10 hours ago + 10 days ago (assumes 1h bars) ---
    for lag in [1, 3, 5, 10]:
        f[f"close_lag_{lag}h"] = close.shift(lag)
        f[f"ret_lag_{lag}h"] = close.pct_change(lag)
    lag_10d = 10 * 24  # 10 days in hourly bars
    f["close_lag_10d"] = close.shift(lag_10d)
    f["ret_lag_10d"] = close.pct_change(lag_10d)

    # --- moving averages & price-vs-MA (12 features) ---
    for w in [5, 10, 20, 50, 100, 200]:
        ma = close.rolling(w).mean()
        f[f"ma_{w}"] = ma
        f[f"px_over_ma_{w}"] = close / ma - 1

    # --- volatility (8 features) ---
    for w in [5, 10, 20, 50]:
        f[f"vol_{w}"] = close.pct_change().rolling(w).std()
    f["atr_14"] = _atr(df, 14)
    f["atr_50"] = _atr(df, 50)
    f["hl_range"] = (high - low) / close
    f["oc_range"] = (close - df["Open"]) / df["Open"]

    # --- Bollinger Bands (4 features) ---
    bb_ma = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    f["bb_upper"] = bb_ma + 2 * bb_std
    f["bb_lower"] = bb_ma - 2 * bb_std
    f["bb_pct_b"] = (close - f["bb_lower"]) / (f["bb_upper"] - f["bb_lower"])
    f["bb_width"] = (f["bb_upper"] - f["bb_lower"]) / bb_ma

    # --- RSI at multiple windows (3 features) ---
    for p in [7, 14, 21]:
        f[f"rsi_{p}"] = _rsi(close, p)

    # --- MACD (3 features) ---
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    f["macd"] = macd
    f["macd_signal"] = macd.ewm(span=9, adjust=False).mean()
    f["macd_hist"] = f["macd"] - f["macd_signal"]

    # --- volume features (5 features) ---
    for w in [5, 10, 20]:
        f[f"vol_ma_{w}"] = vol.rolling(w).mean()
    f["vol_change"] = vol.pct_change()
    f["vol_zscore_20"] = (vol - vol.rolling(20).mean()) / vol.rolling(20).std()

    # --- candle shape (3 features) ---
    f["body"] = (close - df["Open"]).abs() / df["Open"]
    f["upper_wick"] = (high - df[["Open", "Close"]].max(axis=1)) / df["Open"]
    f["lower_wick"] = (df[["Open", "Close"]].min(axis=1) - low) / df["Open"]

    # --- calendar / session features (5 features) ---
    f["hour"] = df.index.hour
    f["dayofweek"] = df.index.dayofweek
    f["is_london_session"] = df.index.hour.isin(range(7, 16)).astype(int)
    f["is_ny_session"] = df.index.hour.isin(range(12, 21)).astype(int)
    f["month"] = df.index.month

    # --- optional external series if present (DXY/VIX/SP500) ---
    for col in df.columns:
        if col.endswith("_close"):
            f[f"{col}_ret_1"] = df[col].pct_change()
            f[f"{col}_ret_5"] = df[col].pct_change(5)

    print(f"[features] built {f.shape[1]} raw features")
    return f


def make_target(df: pd.DataFrame, horizon: int = None) -> pd.Series:
    """Forward return over `horizon` bars -- what the models try to predict."""
    horizon = horizon or TARGET_HORIZON
    return df["Close"].pct_change(horizon).shift(-horizon)


def select_top_features(X: pd.DataFrame, y: pd.Series, n: int = None) -> list:
    """Rank features by RandomForest importance, return top-n column names.
    This is the '50+ features -> select 20-30' step in the diagram."""
    n = n or N_FEATURES_SELECTED
    mask = X.notna().all(axis=1) & y.notna()
    rf = RandomForestRegressor(
        n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1
    )
    rf.fit(X.loc[mask], y.loc[mask])
    importances = pd.Series(rf.feature_importances_, index=X.columns)
    top = importances.sort_values(ascending=False).head(n).index.tolist()
    print(f"[features] selected top {len(top)} features:\n{top}")
    return top


if __name__ == "__main__":
    # เหตุผลที่รันตรงๆแล้วไม่ขึ้นอะไร: ไฟล์นี้เดิมมีแต่ def function ไม่มีจุดเรียกใช้งาน
    # เพิ่ม demo block นี้ไว้ ให้รัน `python features.py` แล้วเห็นผลจริงจาก XAU_1m_data.csv
    import sys

    csv_path = "data/XAU_1m_data.csv" if "ipykernel" in sys.modules else (
        sys.argv[1] if len(sys.argv) > 1 else "XAU_1m_data.csv"
    )

    raw = pd.read_csv(csv_path, parse_dates=["Date"]).set_index("Date").sort_index()
    df_1h = raw.resample("1h").agg(
        {"Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"}
    ).dropna()
    print(f"[features] loaded {len(df_1h):,} hourly bars")

    feats = build_features(df_1h)
    y = make_target(df_1h)

    print("\nshape:", feats.shape)
    print("\nlast 5 rows (lag columns):")
    lag_cols = [c for c in feats.columns if "lag" in c]
    print(feats[lag_cols].tail())

    # select_top_features() ใช้ RF fit บนข้อมูลทั้งหมด (~125k แถว) จะช้าถ้าเครื่องแรงไม่พอ
    # เปิดใช้เองตอนพร้อม:
    # top_feats = select_top_features(feats, y)

[features] loaded 12,230 hourly bars
[features] built 65 raw features

shape: (12230, 65)

last 5 rows (lag columns):
                     close_lag_1h  ret_lag_1h  close_lag_3h  ret_lag_3h  \
Date                                                                      
2026-02-27 01:00:00      5175.100    0.003329       5185.25    0.001365   
2026-02-27 02:00:00      5192.330   -0.001649       5184.76   -0.000191   
2026-02-27 03:00:00      5183.770    0.000040       5175.10    0.001715   
2026-02-27 04:00:00      5183.975   -0.000421       5192.33   -0.002029   
2026-02-27 05:00:00      5181.795    0.002123       5183.77    0.001741   

                     close_lag_5h  ret_lag_5h  close_lag_10h  ret_lag_10h  \
Date                                                                        
2026-02-27 01:00:00       5187.00    0.001028       5164.745     0.005341   
2026-02-27 02:00:00       5196.62   -0.002473       5185.775    -0.000387   
2026-02-27 03:00:00       5185.25   -0.000246   